In [ ]:
pip install yfinance transformers feedparser beautifulsoup4 pandas requests transformers

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 4.3 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=a2c94eec8715f46f2e0df247603f7fb3ed257faa5fd7fa4252f5bcfa35905f53
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import yfinance as yf
import feedparser

SEC Request Headers

In [ ]:
# Required by the SEC to identify the requester.
#This ensures compliant and responsible data access when using SEC's EDGAR API.

HEADERS = {
    "User-Agent": "MarioYanez CaliforniaStateUniversityFresno myanez987@mail.fresnostate.edu",
    "Accept-Encoding": "gzip, deflate"
}

Finbert setup

In [ ]:
#Loads a financial domain version for BERT (FinBERT) for classifying text as positive, negative, or neutral.
tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")
model = AutoModelForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone")
finbert = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: yiyanghkust/finbert-tone
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tickers

In [ ]:
ticker_input = input("Enter comma-separated stock tickers (e.g., NVDA, AAPL, MSFT): ")
tickers = [t.strip().upper() for t in ticker_input.split(",") if t.strip()]

search_terms = {t: t for t in tickers}
print(f"Will analyze: {list(search_terms.keys())}")

Enter comma-separated stock tickers (e.g., NVDA, AAPL, MSFT): MSFT
Will analyze: ['MSFT']


Finviz Scraper

In [ ]:
# Pulling news headlines from Finviz
#Scrapes the latest headlines from the stock's Finviz profile page
def get_finviz_headlines(ticker):
    url = f"https://finviz.com/quote.ashx?t={ticker}"
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        news_table = soup.find("table", class_="fullview-news-outer")
        rows = news_table.find_all("tr") if news_table else []
        headlines = [row.a.get_text(strip=True) for row in rows if row.a]
        print(f"Pulled {len(headlines)} Finviz headlines for {ticker}")
        return headlines
    except Exception as e:
        print(f"Finviz fetch failed for {ticker}: {e}")
        return []

Google News RSS

In [ ]:
#Getting headlines from Google News RSS
#Queries Google News RSS feed using the format [TICKER] stock

def get_google_news_rss(ticker):
    query = f"{ticker} stock"
    url = f"https://news.google.com/rss/search?q={query.replace(' ', '+')}"
    try:
        feed = feedparser.parse(url)
        entries = [entry.title for entry in feed.entries]
        print(f"Pulled {len(entries)} Google RSS headlines for {ticker}")
        return entries
    except Exception as e:
        print(f"Google RSS fetch failed for {ticker}: {e}")
        return []




SEC

Getting CIK Map

In [ ]:
#Pulls public JSON file from the SEC and returns a dictionaty mapping tickers to a 10 digit CIK codes

def get_cik_map():
    url = "https://www.sec.gov/files/company_tickers.json"
    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        data = response.json()
        cik_map = {
            entry["ticker"]: str(entry["cik_str"]).zfill(10)
            for entry in data.values()
        }
        return cik_map
    except Exception as e:
        print(f"Failed to fetch CIK map: {e}")
        return {}
CIK_MAP = get_cik_map()

Get SEC filings

In [ ]:
#Gets last SEC 10-Q Filings
#Pulls and parse the four most recent quarterly filings from EDGAR

def get_recent_sec_filing_texts(ticker, form_type="10-Q", count=2):
    cik = CIK_MAP.get(ticker.upper())
    if not cik:
        print(f"CIK not found for {ticker}")
        return []

    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    print(f"[DEBUG] URL: {url}")
    try:
        res = requests.get(url, headers=HEADERS)
        print(f"[DEBUG] Status Code: {res.status_code}")
        res.raise_for_status()
    except Exception as e:
        print(f"Failed to get filings for {ticker}: {e}")
        return []

    data = res.json()
    filings = data.get("filings", {}).get("recent", {})
    forms = filings.get("form", [])
    documents = filings.get("primaryDocument", [])
    accession_numbers = filings.get("accessionNumber", [])

    indices = [
        i for i, (f, d) in enumerate(zip(forms, documents))
        if "10-q" in f.lower() or "10-q" in d.lower()
    ]

    print(f"[DEBUG] Matching SEC indices: {indices}")
    texts = []

    for i in indices[:count]:
        acc_num = accession_numbers[i].replace("-", "")
        doc_name = documents[i]
        link = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_num}/{doc_name}"
        print(f"[DEBUG] Trying SEC link: {link}")

        try:
            response = requests.get(link, headers=HEADERS)
            soup = BeautifulSoup(response.text, "html.parser")
            text = soup.get_text(separator=" ", strip=True)

            if len(text.strip()) < 100:
              continue

            texts.append(text[:1500])
        except Exception as e:
            print(f"Could not parse SEC filing for {ticker}: {e}")

    print(f"{ticker}: Parsed {len(texts)} {form_type} filings")
    return texts


8K

In [ ]:
def get_recent_sec_filing_texts(ticker, form_type="8-K", count=2):
    cik = CIK_MAP.get(ticker.upper())
    if not cik:
        print(f"CIK not found for {ticker}")
        return []

    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    print(f"[DEBUG] URL: {url}")
    try:
        res = requests.get(url, headers=HEADERS)
        print(f"[DEBUG] Status Code: {res.status_code}")
        res.raise_for_status()
    except Exception as e:
        print(f"Failed to get filings for {ticker}: {e}")
        return []

    data = res.json()
    filings = data.get("filings", {}).get("recent", {})
    forms = filings.get("form", [])
    documents = filings.get("primaryDocument", [])
    accession_numbers = filings.get("accessionNumber", [])

    indices = [
        i for i, f in enumerate(forms)
        if form_type.lower() in f.lower()
    ]

    print(f"[DEBUG] Matching SEC indices: {indices}")
    texts = []

    for i in indices[:count]:
        acc_num = accession_numbers[i].replace("-", "")
        doc_name = documents[i]
        link = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_num}/{doc_name}"
        print(f"[DEBUG] Trying SEC link: {link}")

        try:
            response = requests.get(link, headers=HEADERS)
            soup = BeautifulSoup(response.text, "html.parser")
            text = soup.get_text(separator=" ", strip=True)

            if len(text.strip()) < 100:
                continue

            texts.append(text[:1500])
        except Exception as e:
            print(f"Could not parse SEC filing for {ticker}: {e}")

    print(f"{ticker}: Parsed {len(texts)} {form_type} filings")
    return texts

Finbert Sentiment Analysis

In [ ]:
#Classifies each headline or snippet as poisitive, negative, or neutral
#Converts the result into numneric score for aggregation

def analyze_headlines_with_finbert(headlines):
    sentiments = []
    for headline in headlines:
        try:

            tokens = tokenizer.tokenize(headline)
            if len(tokens) > 512:
                tokens = tokens[:512]
                headline = tokenizer.convert_tokens_to_string(tokens)

            result = finbert(headline)[0]
            label = result["label"].lower()
            score = result["score"]
            print(f"{headline[:60]}... → {label} ({score:.4f})")

            if label == "positive":
                sentiments.append(score)
            elif label == "negative":
                sentiments.append(-score)
            else:
                sentiments.append(0.01)
        except Exception as e:
            print(f"Sentiment error on headline: {headline[:50]} — {e}")
    return sentiments

Main Execution

In [ ]:
def run_pipeline():
    summary_data = []
    for ticker, query in search_terms.items():
        print(f"Processing {ticker}")
        finviz_headlines = get_finviz_headlines(ticker)
        google_headlines = get_google_news_rss(ticker)
        sec_texts = get_recent_sec_filing_texts(ticker)
        combined_texts = finviz_headlines + google_headlines + sec_texts
        if combined_texts:
            sentiment_scores = analyze_headlines_with_finbert(combined_texts)
            avg_score = sum(sentiment_scores) / len(sentiment_scores)
            label = (
                "Buy" if avg_score > 0.1 else
                "Hold" if avg_score >= -0.1 else
                "Sell" if avg_score > -0.4 else
                "Strong Sell"
            )
            summary_data.append({
                "Ticker": ticker,
                "Average Score": avg_score,
                "Sentiment Label": label,
            })
        else:
            print(f"No headlines found for {ticker}")
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_csv("sentiment_regression_data_2025.csv", index=False)
    print("Saved as 'sentiment_regression_data_2025.csv'")
    print(summary_df)
if __name__ == "__main__":
    run_pipeline()

Processing MSFT
Pulled 100 Finviz headlines for MSFT
Pulled 100 Google RSS headlines for MSFT
[DEBUG] URL: https://data.sec.gov/submissions/CIK0000789019.json
[DEBUG] Status Code: 200
[DEBUG] Matching SEC indices: [2, 39, 58, 92, 102, 137, 141, 181, 211, 212, 223, 237, 251, 303, 305, 348, 361, 389, 390, 404, 431, 433, 436, 441, 442, 443, 476, 478, 512, 546, 548, 551, 579, 618, 630, 645, 682, 684, 708, 717, 759, 762, 765, 784, 790, 792, 794, 813, 816, 834, 844, 864, 869, 910, 914, 915, 917, 918, 940, 944, 948, 951, 952, 961, 971, 989, 998]
[DEBUG] Trying SEC link: https://www.sec.gov/Archives/edgar/data/789019/000119312526191457/msft-20260429.htm
[DEBUG] Trying SEC link: https://www.sec.gov/Archives/edgar/data/789019/000119312526027198/msft-20260128.htm
MSFT: Parsed 2 8-K filings
Microsoft Finds Just 13% of Firms Reward AI-Driven Workplace... → neutral (1.0000)
eToro CEO talks Coinbase layoffs & AI's labor force impact... → negative (0.9633)
Alphabet stock climbs 2% on report of massive